In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q -U google-generativeai gitpython nbformat python-docx pypandoc

In [ ]:
import os
import shutil
import git
import nbformat
import google.generativeai as genai
from kaggle_secrets import UserSecretsClient

# Setup Google AI SDK
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("GOOGLE_API_KEY")
genai.configure(api_key=api_key)

# Initialize Model (Gemini 1.5 Pro is recommended for large codebases)
model = genai.GenerativeModel('gemini-2.5-flash-lite')

In [ ]:
import os
import shutil
import git
import nbformat
import google.generativeai as genai
from urllib.parse import urlparse

def extract_code_from_repo(repo_url, target_files=None, local_dir="temp_repo"):
    """
    Clones repo and extracts code. 
    If target_files is provided (list of strings), it ONLY reads those files.
    """
    # --- 1. URL Parsing & Cloning ---
    parsed = urlparse(repo_url)
    path_parts = parsed.path.strip('/').split('/')
    
    # Detect if URL is a tree/master link
    if len(path_parts) > 2 and 'tree' in path_parts:
        tree_index = path_parts.index('tree')
        real_repo_path = "/".join(path_parts[:tree_index])
        git_url = f"https://github.com/{real_repo_path}.git"
        # Calculate subfolder path if it exists
        if len(path_parts) > tree_index + 2:
            target_subfolder = os.path.join(*path_parts[tree_index+2:])
        else:
            target_subfolder = ""
    else:
        git_url = repo_url if repo_url.endswith('.git') else f"{repo_url}.git"
        target_subfolder = ""

    # Clean up temp dir
    if os.path.exists(local_dir):
        shutil.rmtree(local_dir)
    
    try:
        print(f"Cloning {git_url}...")
        git.Repo.clone_from(git_url, local_dir)
    except Exception as e:
        return f"Error cloning: {e}"

    work_dir = os.path.join(local_dir, target_subfolder)
    if not os.path.exists(work_dir):
        return f"Error: Directory '{target_subfolder}' not found."

    code_context = []
    found_files = []

    # --- 2. Walking and Filtering ---
    print(f"Scanning files in {target_subfolder if target_subfolder else 'root'}...")
    
    for root, dirs, files in os.walk(work_dir):
        if '.git' in root:
            continue
            
        for file in files:
            file_path = os.path.join(root, file)
            rel_path = os.path.relpath(file_path, local_dir) 
            
            # Filter Logic: Check if user requested specific files
            if target_files:
                # Check if filename matches OR if full relative path matches
                if file not in target_files and rel_path not in target_files:
                    continue
            
            found_files.append(file)

            # Read Files
            if file.endswith(('.py', '.md', '.txt')):
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        content = f.read()
                        code_context.append(f"--- FILE: {rel_path} ---\n{content}\n")
                except: pass
            
            elif file.endswith('.ipynb'):
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        nb = nbformat.read(f, as_version=4)
                        # Extract code cells
                        cells = [c.source for c in nb.cells if c.cell_type == 'code']
                        
                        # Fix for SyntaxError: Join string outside f-string
                        notebook_content = '\n'.join(cells)
                        code_context.append(f"--- FILE: {rel_path} (Notebook) ---\n{notebook_content}\n")
                except: pass

    # Clean up
    shutil.rmtree(local_dir)
    
    if target_files and not code_context:
        return f"Error: Could not find any of the specified files: {target_files}. Found files: {found_files}"
        
    print(f"Successfully extracted content from: {found_files}")
    return "\n".join(code_context)


def generate_mdt(repo_url, target_files=None):
    """
    Main function to generate MDT.
    Accepts target_files list to filter specific scripts.
    """
    # Pass the list of files to the extraction function
    codebase = extract_code_from_repo(repo_url, target_files=target_files)
    
    if "Error" in codebase:
        return codebase
    
    # Construct Prompt
    prompt = f"""
    {MDT_TEMPLATE_INSTRUCTIONS}
    
    Analyze the specific code files below to generate the documentation.
    
    --- SOURCE CODE ---
    {codebase}
    --- END SOURCE CODE ---
    """
    
    print("Generating MDT...")
    # Generate
    response = model.generate_content(prompt)
    return response.text

In [ ]:
# This is your strict template instruction
MDT_TEMPLATE_INSTRUCTIONS = """
You are a Technical Documentation AI. Your task is to write a Model Documentation Template (MDT) based on the provided code.

Please follow this structure strictly:

   1. First page should have a header 'Model Documentation Template'. In the next line you should write the model name and version. In the next line include current date and model creator's name
   2. Executive Summary
       a. Introduction
       b. Model Purpose, Use and scope
       c. Model Structure and specification
       d. Risk Appetite or threshold for acceptable performance
       e. Model Limitations
       f. Model Impact or Performance Assessment
       g. Model Governance and compliance
    3. Business Requirements
        a. Model Scope, Output and use
        b. Model Network
        c. Model Acceptance Criteria
        d. Model Use Envronment
    4. Model Deevelopment Data
        a. Data Sources
        b. Data Controls
        c. Data Characteristics
        d. Data Preparation
        e. Data Cleaning and Data Treatment
        f. Model Variables
        g. Cration of Datasets, Development, Holdout
    5. Model Methodology
        a. Possible Methodologies
        b. Assessment of Methodologies
        c. Preferred Methodology
    6. Model Specification and Development
        a. Model Variable and Characteristic Selection
        b. Candidate Models
        c. Model Assumptions and Limitations
        d. Model Development Process
    7. Model testing, Test results and Test conclusions
        a. Quantitatve test
        b. Qualitative Tests
        c. Feature Impotance Interpretation
        d. Model Bias Check
        e. Model Calibration and Adjustment
        f. Model Performance on OOT Dataset
        g. WPB model performance 
        e. User Engagement
    8. Final Model
        a. Model Architecture
        b. Model Implementation
        c. Implementaion Controls and Governance
    9. Model risk and Model Monitoring
        a. Model Risks
        b. Model monitoring arrangement
        c. Model gvernance and compliance
        d. Eecutive governance
        e. Review and challange
        f. Compliance with regulatory requirements
        g. Compliance with Model risk FIM and MDT requirements
        e. Regulatory notification and approval
        f. Ongoing governance and control
    10. Appendix
        a. Glossary
        b. GMRS modelling standards
        c. Model compliance
"""

In [ ]:
# 1. Define your repo
repo_link = "https://github.com/nkarchak/Projects/tree/master"

# 2. Define ONLY the files you want the agent to read
# (Change these names to match the actual files in that repo)
my_files = [
    "chicago data analysis.ipynb", 
]

# 3. Run the agent with the filter
documentation = generate_mdt(repo_link, target_files=my_files)

# 4. View
from IPython.display import Markdown
display(Markdown(documentation))

In [ ]:
import pypandoc

# This will download the Pandoc binary into a temporary directory
# and configure pypandoc to use it.
pypandoc.download_pandoc()

# Now your conversion code should work:
pypandoc.convert_text(
    documentation, 
    'docx', 
    format='md', 
    outputfile='Model_Documentation_Final.docx'
)